# CReST: Final Sensitivity and Robustness

Runs final sensitivity analyses for representation choice, bidirectional availability, and Visual-domain expansion.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Final sensitivity / robustness environment

from pathlib import Path
import json

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment


BASE_DIR = Path.cwd()

RESULTS_DIR = (
    BASE_DIR
    / "results"
)

FIG_DIR = (
    BASE_DIR
    / "figures"
)

PROFILE_BEYOND_DIR = (
    RESULTS_DIR
    / "profile_beyond_overall_proficiency"
)

PROFILE_STABILITY_DIR = (
    RESULTS_DIR
    / "crosssystem_profile_stability"
)

SENSITIVITY_DIR = (
    RESULTS_DIR
    / "final_sensitivity_robustness"
)

SENSITIVITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Load permanent profile archive

profile_archive = np.load(
    PROFILE_STABILITY_DIR
    / "crest_oof_primary3_profile_scores.npz",
    allow_pickle=True
)


profile_domains = (
    profile_archive[
        "domains"
    ]
    .astype(str)
    .tolist()
)


profile_complete = (
    profile_archive[
        "profile_complete"
    ]
    .astype(bool)
)


direction_count = (
    profile_archive[
        "direction_count"
    ]
    .astype(
        np.int8
    )
)


relative_profile_primary = (
    profile_archive[
        "relative_profile"
    ]
    .astype(
        np.float32
    )
)


relative_profile_combined = (
    profile_archive[
        "relative_profile_combined_sensitivity"
    ]
    .astype(
        np.float32
    )
)


primary_axes = np.load(
    PROFILE_STABILITY_DIR
    / "crest_continuous_profile_axes.npy"
).astype(
    np.float64
)


assert profile_domains == [
    "Written",
    "Social",
    "Scientific"
]


assert relative_profile_primary.shape == (
    len(
        profile_complete
    ),
    3,
    10
)


assert relative_profile_combined.shape == (
    len(
        profile_complete
    ),
    3,
    10
)


assert primary_axes.shape == (
    3,
    2
)


print(
    "Students:",
    len(
        profile_complete
    )
)

print(
    "Primary profile cohort:",
    int(
        profile_complete.sum()
    )
)

print(
    "Domains:",
    profile_domains
)

print(
    "Locked primary axes:"
)

print(
    primary_axes
)

print(
    "\nSensitivity environment: PASS"
)

Students: 142564
Primary profile cohort: 139894
Domains: ['Written', 'Social', 'Scientific']
Locked primary axes:
[[-0.25849353  0.77449839]
 [ 0.79998205 -0.16338724]
 [-0.54148852 -0.61111116]]

Sensitivity environment: PASS


In [ ]:
# Lock final sensitivity protocol

FINAL_SENSITIVITY_PROTOCOL = {

    "status":
        "LOCKED BEFORE SENSITIVITY RESULTS",

    "primary_result":
        (
            "Two continuous domain-relative profile dimensions "
            "across Written, Social, and Scientific creative-thinking tasks."
        ),

    "check_1":
        {
            "name":
                "Combined-representation sensitivity",

            "question":
                (
                    "Does adding z_general to the domain-specific "
                    "representation materially change the two continuous "
                    "profile dimensions?"
                ),

            "comparison":
                (
                    "Primary domain-specific OOF profile scores versus "
                    "combined z_general + z_domain_specific OOF profile scores."
                )
        },

    "check_2":
        {
            "name":
                "Both-direction complete-case sensitivity",

            "question":
                (
                    "Are the two continuous dimensions preserved when "
                    "analysis is restricted to students with usable "
                    "representations in both A-to-B and B-to-A directions "
                    "for all three primary domains?"
                )
        },

    "check_3":
        {
            "name":
                "Visual-domain sensitivity",

            "question":
                (
                    "When Visual is added as a fourth domain, does the "
                    "primary Written-Social-Scientific structure remain "
                    "embedded in the expanded profile space?"
                ),

            "status":
                "Run after checks 1 and 2"
        },

    "decision_rule":
        (
            "Sensitivity analyses are used only to test preservation "
            "of the locked primary structure; they will not redefine "
            "the primary analysis."
        )
}


with open(
    SENSITIVITY_DIR
    / "final_sensitivity_protocol.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_SENSITIVITY_PROTOCOL,
        f,
        indent=2
    )


print(
    json.dumps(
        FINAL_SENSITIVITY_PROTOCOL,
        indent=2
    )
)

{
  "status": "LOCKED BEFORE SENSITIVITY RESULTS",
  "primary_result": "Two continuous domain-relative profile dimensions across Written, Social, and Scientific creative-thinking tasks.",
  "check_1": {
    "name": "Combined-representation sensitivity",
    "question": "Does adding z_general to the domain-specific representation materially change the two continuous profile dimensions?",
    "comparison": "Primary domain-specific OOF profile scores versus combined z_general + z_domain_specific OOF profile scores."
  },
  "check_2": {
    "name": "Both-direction complete-case sensitivity",
    "question": "Are the two continuous dimensions preserved when analysis is restricted to students with usable representations in both A-to-B and B-to-A directions for all three primary domains?"
  },
  "check_3": {
    "name": "Visual-domain sensitivity",
    "question": "When Visual is added as a fourth domain, does the primary Written-Social-Scientific structure remain embedded in the expanded pro

In [ ]:
# Consensus-axis and alignment helpers

def derive_consensus_axes(
    relative_profile,
    mask
):

    mask = np.asarray(
        mask,
        dtype=bool
    )


    covariances = []


    for pv_j in range(
        10
    ):

        X = (
            relative_profile[
                mask,
                :,
                pv_j
            ]
            .astype(
                np.float64
            )
        )


        assert np.isfinite(
            X
        ).all()


        covariance = np.cov(
            X,
            rowvar=False
        )


        covariances.append(
            covariance
        )


    covariances = np.stack(
        covariances,
        axis=0
    )


    consensus_covariance = (
        covariances.mean(
            axis=0
        )
    )


    eigenvalues, eigenvectors = (
        np.linalg.eigh(
            consensus_covariance
        )
    )


    order = np.argsort(
        eigenvalues
    )[::-1]


    eigenvalues = (
        eigenvalues[
            order
        ]
    )


    eigenvectors = (
        eigenvectors[
            :,
            order
        ]
    )


    axes = (
        eigenvectors[
            :,
            :2
        ]
        .copy()
    )


    nonzero_share = (
        eigenvalues[
            :2
        ]
        /
        eigenvalues[
            :2
        ].sum()
    )


    return {
        "axes":
            axes,

        "eigenvalues":
            eigenvalues,

        "variance_share":
            nonzero_share,

        "covariances":
            covariances,

        "consensus_covariance":
            consensus_covariance
    }


def align_axes_to_reference(
    axes,
    reference_axes
):

    axes = (
        np.asarray(
            axes,
            dtype=np.float64
        )
        .copy()
    )


    reference_axes = np.asarray(
        reference_axes,
        dtype=np.float64
    )


    similarity = np.abs(
        axes.T
        @
        reference_axes
    )


    source_idx, reference_idx = (
        linear_sum_assignment(
            -similarity
        )
    )


    aligned = np.zeros_like(
        reference_axes
    )


    for source_axis, ref_axis in zip(
        source_idx,
        reference_idx
    ):

        v = (
            axes[
                :,
                source_axis
            ]
            .copy()
        )


        if (
            np.dot(
                v,
                reference_axes[
                    :,
                    ref_axis
                ]
            )
            < 0
        ):

            v *= -1


        aligned[
            :,
            ref_axis
        ] = v


    cosines = np.sum(
        aligned
        *
        reference_axes,
        axis=0
    )


    cosines = np.clip(
        cosines,
        -1.0,
        1.0
    )


    angles = np.degrees(
        np.arccos(
            cosines
        )
    )


    return (
        aligned,
        cosines,
        angles
    )


def pv_axis_stability_against_reference(
    covariances,
    reference_axes
):

    rows = []


    for pv_j in range(
        covariances.shape[
            0
        ]
    ):

        values, vectors = np.linalg.eigh(
            covariances[
                pv_j
            ]
        )


        order = np.argsort(
            values
        )[::-1]


        candidate_axes = (
            vectors[
                :,
                order
            ][
                :,
                :2
            ]
        )


        aligned, cosines, angles = (
            align_axes_to_reference(
                candidate_axes,
                reference_axes
            )
        )


        for dimension in range(
            2
        ):

            rows.append(
                {
                    "pv":
                        pv_j + 1,

                    "dimension":
                        dimension + 1,

                    "cosine":
                        float(
                            cosines[
                                dimension
                            ]
                        ),

                    "angle_degrees":
                        float(
                            angles[
                                dimension
                            ]
                        )
                }
            )


    return pd.DataFrame(
        rows
    )


print(
    "Axis helper functions: PASS"
)

Axis helper functions: PASS


In [ ]:
# Sensitivity 1:
# Combined z_general + z_domain_specific representation

combined_result = (
    derive_consensus_axes(
        relative_profile=(
            relative_profile_combined
        ),
        mask=(
            profile_complete
        )
    )
)


(
    combined_axes_aligned,
    combined_cosines,
    combined_angles
) = align_axes_to_reference(

    combined_result[
        "axes"
    ],

    primary_axes
)


combined_pv_stability = (
    pv_axis_stability_against_reference(

        combined_result[
            "covariances"
        ],

        primary_axes
    )
)


combined_loading_df = pd.DataFrame(
    {
        "domain":
            profile_domains,

        "primary_dimension_1":
            primary_axes[
                :,
                0
            ],

        "combined_dimension_1":
            combined_axes_aligned[
                :,
                0
            ],

        "primary_dimension_2":
            primary_axes[
                :,
                1
            ],

        "combined_dimension_2":
            combined_axes_aligned[
                :,
                1
            ]
    }
)


print(
    "COMBINED-REPRESENTATION SENSITIVITY"
)

print(
    "=" * 80
)


print(
    combined_loading_df.to_string(
        index=False
    )
)


print(
    "\nVariance shares"
)

print(
    "Dimension 1:",
    f"{100 * combined_result['variance_share'][0]:.2f}%"
)

print(
    "Dimension 2:",
    f"{100 * combined_result['variance_share'][1]:.2f}%"
)


print(
    "\nAlignment with locked primary axes"
)

for d in range(
    2
):

    print(
        f"Dimension {d + 1}: "
        f"cosine={combined_cosines[d]:.6f}, "
        f"angle={combined_angles[d]:.3f}°"
    )


print(
    "\nPV-specific alignment with primary axes"
)

print(
    combined_pv_stability
    .groupby(
        "dimension"
    )
    .agg(
        mean_cosine=(
            "cosine",
            "mean"
        ),

        minimum_cosine=(
            "cosine",
            "min"
        ),

        mean_angle_deg=(
            "angle_degrees",
            "mean"
        ),

        maximum_angle_deg=(
            "angle_degrees",
            "max"
        )
    )
    .to_string()
)


combined_loading_df.to_csv(
    SENSITIVITY_DIR
    / "combined_representation_axis_loadings.csv",
    index=False
)


combined_pv_stability.to_csv(
    SENSITIVITY_DIR
    / "combined_representation_pv_axis_stability.csv",
    index=False
)

COMBINED-REPRESENTATION SENSITIVITY
    domain  primary_dimension_1  combined_dimension_1  primary_dimension_2  combined_dimension_2
   Written            -0.258494             -0.228213             0.774498              0.783955
    Social             0.799982              0.793031            -0.163387             -0.194339
Scientific            -0.541489             -0.564819            -0.611111             -0.589616

Variance shares
Dimension 1: 69.69%
Dimension 2: 30.31%

Alignment with locked primary axes
Dimension 1: cosine=0.999245, angle=2.226°
Dimension 2: cosine=0.999245, angle=2.226°

PV-specific alignment with primary axes
           mean_cosine  minimum_cosine  mean_angle_deg  maximum_angle_deg
dimension                                                                
1             0.999225        0.998842         2.22587           2.757173
2             0.999225        0.998842         2.22587           2.757173


In [ ]:
# Sensitivity 2:
# Students available in BOTH A->B and B->A directions
# for all three primary domains

both_direction_mask = (
    profile_complete
    &
    (
        direction_count
        == 2
    )
    .all(
        axis=1
    )
)


n_both_direction = int(
    both_direction_mask.sum()
)


print(
    "Both-direction cohort:",
    f"{n_both_direction:,}"
)

print(
    "Percentage of full primary CT cohort:",
    f"{100 * both_direction_mask.mean():.2f}%"
)


assert (
    n_both_direction
    == 52684
), (
    "Both-direction cohort does not match "
    "the locked availability audit."
)


both_result = (
    derive_consensus_axes(
        relative_profile=(
            relative_profile_primary
        ),
        mask=(
            both_direction_mask
        )
    )
)


(
    both_axes_aligned,
    both_cosines,
    both_angles
) = align_axes_to_reference(

    both_result[
        "axes"
    ],

    primary_axes
)


both_pv_stability = (
    pv_axis_stability_against_reference(

        both_result[
            "covariances"
        ],

        primary_axes
    )
)


both_loading_df = pd.DataFrame(
    {
        "domain":
            profile_domains,

        "primary_dimension_1":
            primary_axes[
                :,
                0
            ],

        "both_direction_dimension_1":
            both_axes_aligned[
                :,
                0
            ],

        "primary_dimension_2":
            primary_axes[
                :,
                1
            ],

        "both_direction_dimension_2":
            both_axes_aligned[
                :,
                1
            ]
    }
)


print(
    "\nBOTH-DIRECTION COMPLETE-CASE SENSITIVITY"
)

print(
    "=" * 80
)


print(
    both_loading_df.to_string(
        index=False
    )
)


print(
    "\nVariance shares"
)

print(
    "Dimension 1:",
    f"{100 * both_result['variance_share'][0]:.2f}%"
)

print(
    "Dimension 2:",
    f"{100 * both_result['variance_share'][1]:.2f}%"
)


print(
    "\nAlignment with locked primary axes"
)

for d in range(
    2
):

    print(
        f"Dimension {d + 1}: "
        f"cosine={both_cosines[d]:.6f}, "
        f"angle={both_angles[d]:.3f}°"
    )


print(
    "\nPV-specific alignment with primary axes"
)

print(
    both_pv_stability
    .groupby(
        "dimension"
    )
    .agg(
        mean_cosine=(
            "cosine",
            "mean"
        ),

        minimum_cosine=(
            "cosine",
            "min"
        ),

        mean_angle_deg=(
            "angle_degrees",
            "mean"
        ),

        maximum_angle_deg=(
            "angle_degrees",
            "max"
        )
    )
    .to_string()
)


both_loading_df.to_csv(
    SENSITIVITY_DIR
    / "both_direction_axis_loadings.csv",
    index=False
)


both_pv_stability.to_csv(
    SENSITIVITY_DIR
    / "both_direction_pv_axis_stability.csv",
    index=False
)


np.save(
    SENSITIVITY_DIR
    / "both_direction_mask.npy",
    both_direction_mask
)


print(
    "\nBoth-direction sensitivity: PASS"
)

Both-direction cohort: 52,684
Percentage of full primary CT cohort: 36.95%

BOTH-DIRECTION COMPLETE-CASE SENSITIVITY
    domain  primary_dimension_1  both_direction_dimension_1  primary_dimension_2  both_direction_dimension_2
   Written            -0.258494                   -0.382389             0.774498                    0.721419
    Social             0.799982                    0.815962            -0.163387                   -0.029551
Scientific            -0.541489                   -0.433573            -0.611111                   -0.691868

Variance shares
Dimension 1: 78.93%
Dimension 2: 21.07%

Alignment with locked primary axes
Dimension 1: cosine=0.986374, angle=9.469°
Dimension 2: cosine=0.986374, angle=9.469°

PV-specific alignment with primary axes
           mean_cosine  minimum_cosine  mean_angle_deg  maximum_angle_deg
dimension                                                                
1             0.986371        0.985665        9.469225           9.713149
2    

In [ ]:
# Infrastructure for Visual-domain sensitivity

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


HANDOFF_DIR = (
    BASE_DIR
    / "processed"
    / "crest_v5_handoff"
)

HIDDEN_DOMAIN_DIR = (
    PROFILE_BEYOND_DIR
    / "hidden_target_domain_latents"
)


# Core data

primary_meta = pd.read_pickle(
    HANDOFF_DIR
    / "primary_meta.pkl"
)


profile_meta = pd.read_pickle(
    PROFILE_BEYOND_DIR
    / "primary_cohort_with_pisa_ct_pvs.pkl"
)


credit_tokens = np.load(
    HANDOFF_DIR
    / "credit_tokens.npy"
)


performance_mask = np.load(
    HANDOFF_DIR
    / "performance_mask.npy"
)


item_domain_ids = np.load(
    HANDOFF_DIR
    / "item_domain_ids.npy"
)


CREATIVE_PV_COLS = [
    f"PV{i}CRTH_NC"
    for i in range(
        1,
        11
    )
]


pv_matrix_all = (
    profile_meta[
        CREATIVE_PV_COLS
    ]
    .to_numpy(
        dtype=np.float32
    )
)


country_all = (
    profile_meta[
        "CNT"
    ]
    .astype(str)
    .to_numpy()
)


n_students = len(
    primary_meta
)


# Locked item split

ITEM_SET_A = np.array(
    [
        0, 1, 5, 6, 7, 8, 9, 10,
        12, 15, 17, 20, 24, 27, 28, 31
    ],
    dtype=np.int64
)


ITEM_SET_B = np.array(
    [
        2, 3, 4, 11, 13, 14, 16, 18,
        19, 21, 22, 23, 25, 26, 29, 30
    ],
    dtype=np.int64
)


DIRECTION_SPECS = {

    "A_to_B": {
        "source":
            ITEM_SET_A,

        "target":
            ITEM_SET_B
    },

    "B_to_A": {
        "source":
            ITEM_SET_B,

        "target":
            ITEM_SET_A
    }
}


VISUAL_DOMAIN_ID = 1


# Helpers

def hidden_domain_target(
    row_indices,
    target_items
):

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64
    )

    target_items = np.asarray(
        target_items,
        dtype=np.int64
    )


    tokens = (
        credit_tokens[
            row_indices
        ][
            :,
            target_items
        ]
    )


    mask = (
        performance_mask[
            row_indices
        ][
            :,
            target_items
        ]
    )


    values = (
        tokens.astype(
            np.float32
        )
        -
        1.0
    ) / 2.0


    values = np.where(
        mask,
        values,
        0.0
    )


    n_scored = (
        mask.sum(
            axis=1
        )
    )


    score = (
        values.sum(
            axis=1
        )
        /
        np.maximum(
            n_scored,
            1
        )
    )


    score[
        n_scored == 0
    ] = np.nan


    return (
        score.astype(
            np.float32
        ),
        n_scored.astype(
            np.int16
        )
    )


def proficiency_conditioned_residual_matrix(
    domain_score,
    pv_matrix,
    countries,
    minimum_system_n=30
):

    domain_score = np.asarray(
        domain_score,
        dtype=float
    )

    pv_matrix = np.asarray(
        pv_matrix,
        dtype=float
    )

    countries = np.asarray(
        countries
    )


    residuals = np.full(
        pv_matrix.shape,
        np.nan,
        dtype=np.float32
    )


    for country in np.unique(
        countries
    ):

        idx = np.flatnonzero(
            countries
            ==
            country
        )


        idx = idx[
            np.isfinite(
                domain_score[
                    idx
                ]
            )
        ]


        if len(
            idx
        ) < minimum_system_n:

            continue


        y = (
            domain_score[
                idx
            ]
            .astype(float)
        )


        for pv_j in range(
            10
        ):

            x = (
                pv_matrix[
                    idx,
                    pv_j
                ]
                .astype(float)
            )


            valid = (
                np.isfinite(x)
                &
                np.isfinite(y)
            )


            if valid.sum() < minimum_system_n:

                continue


            xv = x[
                valid
            ]

            yv = y[
                valid
            ]


            X = np.column_stack(
                [
                    np.ones(
                        len(xv)
                    ),
                    xv
                ]
            )


            beta, _, _, _ = (
                np.linalg.lstsq(
                    X,
                    yv,
                    rcond=None
                )
            )


            resid = (
                yv
                -
                X @ beta
            )


            sd = (
                resid.std(
                    ddof=1
                )
            )


            if (
                not np.isfinite(sd)
                or
                sd <= 0
            ):

                continue


            residuals[
                idx[
                    valid
                ],
                pv_j
            ] = (
                resid
                /
                sd
            ).astype(
                np.float32
            )


    return residuals


def fit_multitarget_ridge(
    X_train,
    Y_train,
    X_test
):

    valid = (
        np.isfinite(
            X_train
        ).all(
            axis=1
        )
        &
        np.isfinite(
            Y_train
        ).all(
            axis=1
        )
    )


    scaler = StandardScaler()


    X_train_scaled = (
        scaler.fit_transform(
            X_train[
                valid
            ]
        )
    )


    X_test_scaled = (
        scaler.transform(
            X_test
        )
    )


    probe = Ridge(
        alpha=1.0
    )


    probe.fit(
        X_train_scaled,
        Y_train[
            valid
        ]
    )


    return (
        probe.predict(
            X_test_scaled
        )
        .astype(
            np.float32
        )
    )


print(
    "Visual sensitivity infrastructure: PASS"
)

Visual sensitivity infrastructure: PASS


In [ ]:
# Build OOF Visual-domain proficiency-conditioned predictions

visual_pred_direction = {

    "A_to_B":
        np.full(
            (
                n_students,
                10
            ),
            np.nan,
            dtype=np.float32
        ),

    "B_to_A":
        np.full(
            (
                n_students,
                10
            ),
            np.nan,
            dtype=np.float32
        )
}


visual_available_direction = {

    "A_to_B":
        np.zeros(
            n_students,
            dtype=bool
        ),

    "B_to_A":
        np.zeros(
            n_students,
            dtype=bool
        )
}


visual_target_item_count = {}


for direction, spec in (
    DIRECTION_SPECS.items()
):

    visual_target_item_count[
        direction
    ] = int(
        sum(
            item_domain_ids[
                i
            ]
            ==
            VISUAL_DOMAIN_ID

            for i
            in spec[
                "target"
            ]
        )
    )


print(
    "Visual target-item counts:",
    visual_target_item_count
)


# Fold-wise validation -> unseen test prediction

for outer_fold in range(
    7
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"VISUAL OOF PROFILE — FOLD "
        f"{outer_fold}"
    )

    print(
        "=" * 72
    )


    for direction, spec in (
        DIRECTION_SPECS.items()
    ):

        val_data = np.load(
            HIDDEN_DOMAIN_DIR
            /
            (
                f"fold_{outer_fold}_"
                f"validation_"
                f"{direction}.npz"
            )
        )


        test_data = np.load(
            HIDDEN_DOMAIN_DIR
            /
            (
                f"fold_{outer_fold}_"
                f"test_"
                f"{direction}.npz"
            )
        )


        val_idx = (
            val_data[
                "handoff_row_index"
            ]
            .astype(
                np.int64
            )
        )


        test_idx = (
            test_data[
                "handoff_row_index"
            ]
            .astype(
                np.int64
            )
        )


        zd_val = (
            val_data[
                "z_domain_specific"
            ][
                :,
                VISUAL_DOMAIN_ID,
                :
            ]
        )


        zd_test = (
            test_data[
                "z_domain_specific"
            ][
                :,
                VISUAL_DOMAIN_ID,
                :
            ]
        )


        da_val = (
            val_data[
                "domain_available"
            ][
                :,
                VISUAL_DOMAIN_ID
            ]
            .astype(bool)
        )


        da_test = (
            test_data[
                "domain_available"
            ][
                :,
                VISUAL_DOMAIN_ID
            ]
            .astype(bool)
        )


        target_items = np.array(
            [
                i
                for i in spec[
                    "target"
                ]
                if (
                    item_domain_ids[
                        i
                    ]
                    ==
                    VISUAL_DOMAIN_ID
                )
            ],
            dtype=np.int64
        )


        assert len(
            target_items
        ) == 2


        # Validation target only

        (
            y_val,
            n_target_val
        ) = hidden_domain_target(
            val_idx,
            target_items
        )


        eligible_train = (
            da_val
            &
            (
                n_target_val
                >= 1
            )
        )


        Y_val = (
            proficiency_conditioned_residual_matrix(

                domain_score=(
                    y_val[
                        eligible_train
                    ]
                ),

                pv_matrix=(
                    pv_matrix_all[
                        val_idx[
                            eligible_train
                        ]
                    ]
                ),

                countries=(
                    country_all[
                        val_idx[
                            eligible_train
                        ]
                    ]
                ),

                minimum_system_n=30
            )
        )


        train_complete = (
            np.isfinite(
                Y_val
            )
            .all(
                axis=1
            )
        )


        X_train = (
            zd_val[
                eligible_train
            ][
                train_complete
            ]
        )


        Y_train = (
            Y_val[
                train_complete
            ]
        )


        # Predict unseen test students using source-side
        # Visual representation only.

        eligible_test = (
            da_test
        )


        test_rows = (
            test_idx[
                eligible_test
            ]
        )


        X_test = (
            zd_test[
                eligible_test
            ]
        )


        pred = (
            fit_multitarget_ridge(
                X_train,
                Y_train,
                X_test
            )
        )


        visual_pred_direction[
            direction
        ][
            test_rows
        ] = pred


        visual_available_direction[
            direction
        ][
            test_rows
        ] = True


        print(
            f"{direction}: "
            f"train={len(Y_train):,}, "
            f"test predicted={len(test_rows):,}"
        )


# Combine A->B and B->A

visual_prediction = np.full(
    (
        n_students,
        10
    ),
    np.nan,
    dtype=np.float32
)


visual_direction_count = np.zeros(
    n_students,
    dtype=np.int8
)


numerator = np.zeros(
    (
        n_students,
        10
    ),
    dtype=np.float64
)


denominator = np.zeros(
    n_students,
    dtype=np.float64
)


for direction in [
    "A_to_B",
    "B_to_A"
]:

    available = (
        visual_available_direction[
            direction
        ]
    )


    weight = float(
        visual_target_item_count[
            direction
        ]
    )


    numerator[
        available
    ] += (
        weight
        *
        visual_pred_direction[
            direction
        ][
            available
        ]
    )


    denominator[
        available
    ] += weight


    visual_direction_count[
        available
    ] += 1


visual_available = (
    denominator
    >
    0
)


visual_prediction[
    visual_available
] = (

    numerator[
        visual_available
    ]

    /

    denominator[
        visual_available,
        None
    ]

).astype(
    np.float32
)


four_domain_complete = (
    profile_complete
    &
    visual_available
)


print(
    "\nVisual available:",
    f"{visual_available.sum():,}",
    f"({100 * visual_available.mean():.2f}%)"
)


print(
    "Complete Written + Social + Scientific + Visual:",
    f"{four_domain_complete.sum():,}",
    f"({100 * four_domain_complete.mean():.2f}%)"
)


assert (
    four_domain_complete.sum()
    == 94815
), (
    "Four-domain cohort does not match "
    "the locked availability audit."
)


np.savez_compressed(

    SENSITIVITY_DIR
    / "visual_oof_profile_predictions.npz",

    visual_prediction=(
        visual_prediction
    ),

    visual_available=(
        visual_available
    ),

    visual_direction_count=(
        visual_direction_count
    ),

    four_domain_complete=(
        four_domain_complete
    )
)


print(
    "\nVisual OOF prediction archive: PASS"
)

Visual target-item counts: {'A_to_B': 2, 'B_to_A': 2}

VISUAL OOF PROFILE — FOLD 0
A_to_B: train=1,786, test predicted=8,311
B_to_A: train=1,830, test predicted=8,513

VISUAL OOF PROFILE — FOLD 1
A_to_B: train=1,879, test predicted=7,746
B_to_A: train=1,901, test predicted=8,233

VISUAL OOF PROFILE — FOLD 2
A_to_B: train=1,924, test predicted=7,831
B_to_A: train=1,949, test predicted=8,262

VISUAL OOF PROFILE — FOLD 3
A_to_B: train=1,860, test predicted=7,870
B_to_A: train=1,870, test predicted=8,120

VISUAL OOF PROFILE — FOLD 4
A_to_B: train=1,806, test predicted=7,380
B_to_A: train=1,820, test predicted=7,754

VISUAL OOF PROFILE — FOLD 5
A_to_B: train=1,831, test predicted=7,553
B_to_A: train=1,857, test predicted=7,816

VISUAL OOF PROFILE — FOLD 6
A_to_B: train=2,007, test predicted=7,485
B_to_A: train=1,998, test predicted=7,895

Visual available: 96,988 (68.03%)
Complete Written + Social + Scientific + Visual: 94,815 (66.51%)

Visual OOF prediction archive: PASS


In [ ]:
# Sensitivity 3:
# Does the primary 3-domain subspace survive adding Visual?

FOUR_DOMAIN_ORDER = [
    "Written",
    "Social",
    "Scientific",
    "Visual"
]


# 1. Construct four-domain relative profiles

four_domain_profile = np.full(
    (
        n_students,
        4,
        10
    ),
    np.nan,
    dtype=np.float32
)


four_domain_profile[
    :,
    0:3,
    :
] = (
    profile_archive[
        "domain_prediction"
    ]
)


four_domain_profile[
    :,
    3,
    :
] = (
    visual_prediction
)


rows4 = np.flatnonzero(
    four_domain_complete
)


for pv_j in range(
    10
):

    student_mean = (
        four_domain_profile[
            rows4,
            :,
            pv_j
        ]
        .mean(
            axis=1,
            keepdims=True
        )
    )


    four_domain_profile[
        rows4,
        :,
        pv_j
    ] -= student_mean


max_zero_sum_error = float(
    np.max(
        np.abs(
            four_domain_profile[
                rows4
            ]
            .sum(
                axis=1
            )
        )
    )
)


print(
    "Max four-domain zero-sum error:",
    max_zero_sum_error
)


# 2. Embed locked primary 2D axes in 4D

primary_axes_4d = np.zeros(
    (
        4,
        2
    ),
    dtype=np.float64
)


primary_axes_4d[
    0:3,
    :
] = primary_axes


# Still orthonormal and zero-sum
assert np.allclose(
    primary_axes_4d.T
    @
    primary_axes_4d,
    np.eye(2),
    atol=1e-10
)


assert np.allclose(
    primary_axes_4d.sum(
        axis=0
    ),
    0.0,
    atol=1e-10
)


# 3. Consensus four-domain covariance

cov4_by_pv = []


for pv_j in range(
    10
):

    X = (
        four_domain_profile[
            rows4,
            :,
            pv_j
        ]
        .astype(
            np.float64
        )
    )


    cov4_by_pv.append(
        np.cov(
            X,
            rowvar=False
        )
    )


cov4_by_pv = np.stack(
    cov4_by_pv,
    axis=0
)


cov4 = (
    cov4_by_pv.mean(
        axis=0
    )
)


eigvals4, eigvecs4 = np.linalg.eigh(
    cov4
)


order4 = np.argsort(
    eigvals4
)[::-1]


eigvals4 = eigvals4[
    order4
]


eigvecs4 = eigvecs4[
    :,
    order4
]


# Three non-zero dimensions in four-domain zero-sum space
axes4_top3 = (
    eigvecs4[
        :,
        :3
    ]
)


axes4_top2 = (
    eigvecs4[
        :,
        :2
    ]
)


variance_share4 = (
    eigvals4[
        :3
    ]
    /
    eigvals4[
        :3
    ].sum()
)


# 4. Principal angles:
# locked primary 2D subspace vs expanded top-2 subspace

singular_values = np.linalg.svd(
    primary_axes_4d.T
    @
    axes4_top2,
    compute_uv=False
)


singular_values = np.clip(
    singular_values,
    -1.0,
    1.0
)


principal_angles = np.degrees(
    np.arccos(
        singular_values
    )
)


# 5. How much expanded-profile variance lies in the
#    original primary 2D subspace?

variance_in_primary_subspace = float(
    np.trace(
        primary_axes_4d.T
        @
        cov4
        @
        primary_axes_4d
    )
)


total_nonzero_variance4 = float(
    eigvals4[
        :3
    ].sum()
)


primary_subspace_variance_share = (
    variance_in_primary_subspace
    /
    total_nonzero_variance4
)


# 6. Third expanded axis loadings:
# is new variance mainly Visual-related?

third_axis = (
    axes4_top3[
        :,
        2
    ]
    .copy()
)


# orient Visual positive
if (
    third_axis[
        3
    ]
    < 0
):

    third_axis *= -1


expanded_axis_df = pd.DataFrame(
    {
        "domain":
            FOUR_DOMAIN_ORDER,

        "expanded_dimension_1":
            axes4_top3[
                :,
                0
            ],

        "expanded_dimension_2":
            axes4_top3[
                :,
                1
            ],

        "expanded_dimension_3":
            third_axis
    }
)


# 7. PV-specific principal-angle stability

pv_visual_rows = []


for pv_j in range(
    10
):

    vals, vecs = np.linalg.eigh(
        cov4_by_pv[
            pv_j
        ]
    )


    order = np.argsort(
        vals
    )[::-1]


    top2 = (
        vecs[
            :,
            order
        ][
            :,
            :2
        ]
    )


    s = np.linalg.svd(
        primary_axes_4d.T
        @
        top2,
        compute_uv=False
    )


    s = np.clip(
        s,
        -1.0,
        1.0
    )


    angles = np.sort(
        np.degrees(
            np.arccos(
                s
            )
        )
    )


    pv_visual_rows.append(
        {
            "pv":
                pv_j + 1,

            "principal_angle_1":
                float(
                    angles[
                        0
                    ]
                ),

            "principal_angle_2":
                float(
                    angles[
                        1
                    ]
                )
        }
    )


pv_visual_stability = pd.DataFrame(
    pv_visual_rows
)


# Output

print(
    "\nFOUR-DOMAIN EXPANDED VARIANCE STRUCTURE"
)

print(
    "=" * 80
)


print(
    "Dimension 1:",
    f"{100 * variance_share4[0]:.2f}%"
)

print(
    "Dimension 2:",
    f"{100 * variance_share4[1]:.2f}%"
)

print(
    "Dimension 3:",
    f"{100 * variance_share4[2]:.2f}%"
)


print(
    "\nPrincipal angles between locked primary 2D subspace "
    "and expanded top-2 subspace:"
)

print(
    f"{principal_angles[0]:.3f}°, "
    f"{principal_angles[1]:.3f}°"
)


print(
    "\nExpanded-profile variance contained in "
    "locked primary 2D subspace:"
)

print(
    f"{100 * primary_subspace_variance_share:.2f}%"
)


print(
    "\nEXPANDED AXIS LOADINGS"
)

print(
    expanded_axis_df.to_string(
        index=False
    )
)


print(
    "\nPV-SPECIFIC PRINCIPAL ANGLES"
)

print(
    pv_visual_stability.to_string(
        index=False
    )
)


print(
    "\nPV principal-angle summary"
)

print(
    pv_visual_stability[
        [
            "principal_angle_1",
            "principal_angle_2"
        ]
    ]
    .agg(
        [
            "mean",
            "max"
        ]
    )
    .to_string()
)


# Save

expanded_axis_df.to_csv(
    SENSITIVITY_DIR
    / "four_domain_expanded_axis_loadings.csv",
    index=False
)


pv_visual_stability.to_csv(
    SENSITIVITY_DIR
    / "four_domain_pv_subspace_stability.csv",
    index=False
)


np.save(
    SENSITIVITY_DIR
    / "four_domain_consensus_axes.npy",
    axes4_top3
)


print(
    "\nVisual-domain sensitivity analysis: PASS"
)

Max four-domain zero-sum error: 2.980232238769531e-07

FOUR-DOMAIN EXPANDED VARIANCE STRUCTURE
Dimension 1: 47.05%
Dimension 2: 37.25%
Dimension 3: 15.70%

Principal angles between locked primary 2D subspace and expanded top-2 subspace:
0.000°, 87.519°

Expanded-profile variance contained in locked primary 2D subspace:
53.01%

EXPANDED AXIS LOADINGS
    domain  expanded_dimension_1  expanded_dimension_2  expanded_dimension_3
   Written             -0.255302              0.283461             -0.777477
    Social             -0.294211             -0.804916              0.124699
Scientific             -0.315700              0.521306              0.615283
    Visual              0.865213              0.000150              0.037495

PV-SPECIFIC PRINCIPAL ANGLES
 pv  principal_angle_1  principal_angle_2
  1           0.000000          87.798541
  2           0.000002          87.544175
  3           0.000002          87.450194
  4           0.000002          87.233982
  5           0.000002 

In [ ]:
# Resolve Visual sensitivity by matching locked primary
# axes to all three expanded four-domain axes

from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd


# Locked primary axes embedded in four-domain space

primary_axes_4d = np.zeros(
    (
        4,
        2
    ),
    dtype=np.float64
)

primary_axes_4d[
    0:3,
    :
] = primary_axes


# All three non-zero expanded axes

expanded_axes_3d = (
    axes4_top3.copy()
)


# Absolute cosine similarity matrix
#
# rows    = locked primary dimensions
# columns = expanded dimensions

axis_similarity = np.abs(
    primary_axes_4d.T
    @
    expanded_axes_3d
)


similarity_df = pd.DataFrame(
    axis_similarity,
    index=[
        "Primary Dimension 1",
        "Primary Dimension 2"
    ],
    columns=[
        "Expanded Dimension 1",
        "Expanded Dimension 2",
        "Expanded Dimension 3"
    ]
)


print(
    "ABSOLUTE COSINE SIMILARITY"
)

print(
    "=" * 80
)

print(
    similarity_df.to_string()
)


# Best one-to-one matching:
# two primary dimensions to three expanded dimensions

row_idx, col_idx = (
    linear_sum_assignment(
        -axis_similarity
    )
)


matching_rows = []


for primary_j, expanded_j in zip(
    row_idx,
    col_idx
):

    signed_cosine = float(
        np.dot(
            primary_axes_4d[
                :,
                primary_j
            ],
            expanded_axes_3d[
                :,
                expanded_j
            ]
        )
    )


    absolute_cosine = abs(
        signed_cosine
    )


    angle = float(
        np.degrees(
            np.arccos(
                np.clip(
                    absolute_cosine,
                    -1.0,
                    1.0
                )
            )
        )
    )


    matching_rows.append(
        {
            "primary_dimension":
                primary_j + 1,

            "expanded_dimension":
                expanded_j + 1,

            "signed_cosine":
                signed_cosine,

            "absolute_cosine":
                absolute_cosine,

            "angle_degrees":
                angle
        }
    )


primary_to_expanded_match = pd.DataFrame(
    matching_rows
)


print(
    "\nPRIMARY -> EXPANDED AXIS MATCHING"
)

print(
    "=" * 80
)

print(
    primary_to_expanded_match.to_string(
        index=False
    )
)


# Identify the expanded axis not used by the two
# locked primary dimensions

matched_expanded = set(
    primary_to_expanded_match[
        "expanded_dimension"
    ]
    .astype(int)
    .tolist()
)


unmatched_expanded = [
    d
    for d in [
        1,
        2,
        3
    ]
    if d not in matched_expanded
]


assert len(
    unmatched_expanded
) == 1


visual_axis_id = (
    unmatched_expanded[
        0
    ]
)


visual_axis_vector = (
    expanded_axes_3d[
        :,
        visual_axis_id - 1
    ]
    .copy()
)


# Orient Visual loading positive
if (
    visual_axis_vector[
        3
    ]
    < 0
):

    visual_axis_vector *= -1


visual_axis_loading_df = pd.DataFrame(
    {
        "domain":
            FOUR_DOMAIN_ORDER,

        "loading":
            visual_axis_vector
    }
)


print(
    "\nADDITIONAL VISUAL-SENSITIVITY AXIS"
)

print(
    "=" * 80
)

print(
    "Expanded dimension:",
    visual_axis_id
)


print(
    visual_axis_loading_df.to_string(
        index=False
    )
)


# Variance share of the additional expanded dimension

visual_axis_variance_share = float(
    variance_share4[
        visual_axis_id - 1
    ]
)


print(
    "\nVariance share of additional axis:",
    f"{100 * visual_axis_variance_share:.2f}%"
)


# Save

similarity_df.to_csv(
    SENSITIVITY_DIR
    / "primary_vs_expanded_axis_similarity.csv"
)


primary_to_expanded_match.to_csv(
    SENSITIVITY_DIR
    / "primary_to_expanded_axis_matching.csv",
    index=False
)


visual_axis_loading_df.to_csv(
    SENSITIVITY_DIR
    / "additional_visual_axis_loadings.csv",
    index=False
)


print(
    "\nExpanded-space axis matching: PASS"
)

ABSOLUTE COSINE SIMILARITY
                     Expanded Dimension 1  Expanded Dimension 2  Expanded Dimension 3
Primary Dimension 1              0.001579              0.999472              0.032439
Primary Dimension 2              0.043267              0.032477              0.998536

PRIMARY -> EXPANDED AXIS MATCHING
 primary_dimension  expanded_dimension  signed_cosine  absolute_cosine  angle_degrees
                 1                   2      -0.999472         0.999472       1.861161
                 2                   3       0.998536         0.998536       3.101203

ADDITIONAL VISUAL-SENSITIVITY AXIS
Expanded dimension: 1
    domain   loading
   Written -0.255302
    Social -0.294211
Scientific -0.315700
    Visual  0.865213

Variance share of additional axis: 47.05%

Expanded-space axis matching: PASS


In [ ]:
# Lock final sensitivity and robustness conclusions

import json


match_d1 = (
    primary_to_expanded_match[
        primary_to_expanded_match[
            "primary_dimension"
        ]
        == 1
    ]
    .iloc[
        0
    ]
)


match_d2 = (
    primary_to_expanded_match[
        primary_to_expanded_match[
            "primary_dimension"
        ]
        == 2
    ]
    .iloc[
        0
    ]
)


FINAL_SENSITIVITY_FINDINGS = {

    "status":
        "LOCKED",

    "combined_representation":
        {
            "dimension_1_cosine":
                float(
                    combined_cosines[
                        0
                    ]
                ),

            "dimension_2_cosine":
                float(
                    combined_cosines[
                        1
                    ]
                ),

            "angle_degrees":
                float(
                    combined_angles[
                        0
                    ]
                ),

            "variance_share_dimension_1_percent":
                float(
                    100
                    *
                    combined_result[
                        "variance_share"
                    ][
                        0
                    ]
                ),

            "variance_share_dimension_2_percent":
                float(
                    100
                    *
                    combined_result[
                        "variance_share"
                    ][
                        1
                    ]
                ),

            "conclusion":
                (
                    "The locked two-dimensional profile structure is "
                    "essentially unchanged when z_general is added to "
                    "z_domain_specific."
                )
        },

    "both_direction_complete_case":
        {
            "n_students":
                int(
                    n_both_direction
                ),

            "dimension_1_cosine":
                float(
                    both_cosines[
                        0
                    ]
                ),

            "dimension_2_cosine":
                float(
                    both_cosines[
                        1
                    ]
                ),

            "angle_degrees":
                float(
                    both_angles[
                        0
                    ]
                ),

            "variance_share_dimension_1_percent":
                float(
                    100
                    *
                    both_result[
                        "variance_share"
                    ][
                        0
                    ]
                ),

            "variance_share_dimension_2_percent":
                float(
                    100
                    *
                    both_result[
                        "variance_share"
                    ][
                        1
                    ]
                ),

            "conclusion":
                (
                    "The same qualitative two-axis structure is preserved "
                    "under the stricter both-direction complete-case cohort, "
                    "with moderate rotation and a stronger first dimension."
                )
        },

    "visual_domain_sensitivity":
        {
            "four_domain_complete_n":
                int(
                    four_domain_complete.sum()
                ),

            "four_domain_coverage_percent":
                float(
                    100
                    *
                    four_domain_complete.mean()
                ),

            "expanded_variance_shares_percent":
                [
                    float(
                        100
                        *
                        x
                    )
                    for x in variance_share4
                ],

            "primary_dimension_1_matched_expanded_dimension":
                int(
                    match_d1[
                        "expanded_dimension"
                    ]
                ),

            "primary_dimension_1_absolute_cosine":
                float(
                    match_d1[
                        "absolute_cosine"
                    ]
                ),

            "primary_dimension_1_angle_degrees":
                float(
                    match_d1[
                        "angle_degrees"
                    ]
                ),

            "primary_dimension_2_matched_expanded_dimension":
                int(
                    match_d2[
                        "expanded_dimension"
                    ]
                ),

            "primary_dimension_2_absolute_cosine":
                float(
                    match_d2[
                        "absolute_cosine"
                    ]
                ),

            "primary_dimension_2_angle_degrees":
                float(
                    match_d2[
                        "angle_degrees"
                    ]
                ),

            "additional_axis":
                int(
                    visual_axis_id
                ),

            "additional_axis_variance_share_percent":
                float(
                    100
                    *
                    visual_axis_variance_share
                ),

            "additional_axis_visual_loading":
                float(
                    visual_axis_vector[
                        3
                    ]
                ),

            "conclusion":
                (
                    "Adding Visual does not erase the two locked primary "
                    "dimensions. They reappear almost unchanged among the "
                    "three non-zero expanded dimensions, while Visual adds "
                    "a strong additional contrast that changes eigenvalue "
                    "ordering."
                )
        },

    "overall_conclusion":
        (
            "The continuous Written-Social-Scientific profile structure "
            "is robust to representation choice and stricter bidirectional "
            "availability. Adding Visual expands rather than invalidates "
            "the profile space: the two locked primary dimensions remain "
            "embedded almost unchanged, alongside an additional strongly "
            "Visual-related dimension."
        )
}


with open(
    SENSITIVITY_DIR
    / "crest_final_sensitivity_findings_LOCKED.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_SENSITIVITY_FINDINGS,
        f,
        indent=2
    )


print(
    json.dumps(
        FINAL_SENSITIVITY_FINDINGS,
        indent=2
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "FINAL SENSITIVITY AND ROBUSTNESS: LOCKED"
)

print(
    "=" * 80
)

{
  "status": "LOCKED",
  "combined_representation": {
    "dimension_1_cosine": 0.9992452426166398,
    "dimension_2_cosine": 0.9992452426166398,
    "angle_degrees": 2.226222843037121,
    "variance_share_dimension_1_percent": 69.68923474838255,
    "variance_share_dimension_2_percent": 30.310765251617443,
    "conclusion": "The locked two-dimensional profile structure is essentially unchanged when z_general is added to z_domain_specific."
  },
  "both_direction_complete_case": {
    "n_students": 52684,
    "dimension_1_cosine": 0.9863743852396145,
    "dimension_2_cosine": 0.9863743852396143,
    "angle_degrees": 9.46912930610657,
    "variance_share_dimension_1_percent": 78.93235814227502,
    "variance_share_dimension_2_percent": 21.067641857724983,
    "conclusion": "The same qualitative two-axis structure is preserved under the stricter both-direction complete-case cohort, with moderate rotation and a stronger first dimension."
  },
  "visual_domain_sensitivity": {
    "four_do